In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

In [2]:
%load_ext cudf.pandas

# Students performance in exams
#### Marks secured by the students in college

## Aim
#### To understand the influence of various factors like economic, personal and social on the students performance 

## Inferences would be : 
#### 1. How to imporve the students performance in each test ?
#### 2. What are the major factors influencing the test scores ?
#### 3. Effectiveness of test preparation course?
#### 4. Other inferences 

#### Import the required libraries

In [3]:
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#### Let us initialize the required values ( we will use them later in the program )
#### we will set the minimum marks to 40 to pass in a exam

In [4]:
passmark = 40

In [5]:
import time
start_time = time.time()

#### Let us read the data from the csv file

In [6]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")

# -- STEFANOS -- Replicate Data

In [7]:
### cell 1 ###

factor = 1000
df = pd.concat([df]*factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  object
 6   reading score                1000000 non-null  object
 7   writing score                1000000 non-null  object
dtypes: object(8)
memory usage: 83.8+ MB


#### We will print top few rows to understand about the various data columns

In [8]:
### cell 2 ###

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


#### Size of data frame

In [9]:
### cell 3 ###

print (df.shape)

(1000000, 8)


#### Let us understand about the basic information of the data, like min, max, mean and standard deviation etc.

In [10]:
### cell 4 ###

df.describe()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
unique,2,5,6,2,2,81,72,77
top,female,group C,some college,standard,none,65,72,74
freq,518000,319000,226000,645000,642000,36000,34000,35000


#### Let us check for any missing values

In [11]:
### cell 5 ###

df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

##### As seen above, there are no missing ( null ) values in this dataframe but in real scenarios we need work on dataset with a lot of missing values  

####  Let us explore the Math Score first

In [12]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="math score", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### How many students passed in Math exam ?

In [13]:
### cell 6 ###

df['math score'] = pd.to_numeric(df['math score'], errors='coerce')
df['Math_PassStatus'] = (df['math score'] >= passmark).astype('object').replace({True: 'P', False: 'F'})
df.Math_PassStatus.value_counts()

Math_PassStatus
P    960000
F     40000
Name: count, dtype: int64

In [14]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Math_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Let us explore the Reading score

In [15]:
# STEFANOS: Disable plotting
# sns.countplot(x="reading score", data = df, palette="muted")
# plt.show()

#### How many studends passed in reading ?

In [16]:
### cell 7 ###

df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype('str').replace({'True': 'P', 'False': 'F'})
df.Reading_PassStatus.value_counts()

Reading_PassStatus
P    974000
F     26000
Name: count, dtype: int64

In [17]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Reading_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Let us explore writing score

In [18]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="writing score", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### How many students passed writing ?

In [19]:
### cell 8 ###

df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
df.Writing_PassStatus.value_counts()

Writing_PassStatus
P    968000
F     32000
Name: count, dtype: int64

In [20]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Writing_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Iet us check "How many students passed in all the subjects ?"

In [21]:
### cell 9 ###

df['OverAll_PassStatus'] = ((df['Math_PassStatus'] != 'F') & (df['Reading_PassStatus'] != 'F') & (df['Writing_PassStatus'] != 'F')).astype('str').replace({'True': 'P', 'False': 'F'})
df.OverAll_PassStatus.value_counts()

OverAll_PassStatus
P    949000
F     51000
Name: count, dtype: int64

In [22]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='OverAll_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Find the percentage of marks

In [23]:
### cell 10 ###

df['Total_Marks'] = df[['math score', 'reading score', 'writing score']].sum(axis=1)
df['Percentage'] = df['Total_Marks'] / 3

In [24]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="Percentage", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=0) 

#### Let us assign the grades

### Grading 
####    above 80 = A Grade
####      70 to 80 = B Grade
####      60 to 70 = C Grade
####      50 to 60 = D Grade
####      40 to 50 = E Grade
####    below 40 = F Grade  ( means Fail )


In [25]:
end_time = time.time()
print(end_time - start_time)

8.373480796813965


In [26]:
### cell 11 ###
conditions = [
    (df['OverAll_PassStatus'] == 'F'),
    (df['Percentage'] >= 80),
    (df['Percentage'] >= 70),
    (df['Percentage'] >= 60),
    (df['Percentage'] >= 50),
    (df['Percentage'] >= 40)
]

choices = ['F', 'A', 'B', 'C', 'D', 'E']

df['Grade'] = pd.Series(pd.cut(df['Percentage'],
                                bins=[-float('inf'), 39.999, 49.999, 59.999, 69.999, 79.999, float('inf')],
                                labels=['F', 'E', 'D', 'C', 'B', 'A'])).astype(str)

df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'

df.Grade.value_counts()

Grade
B    261000
C    256000
A    198000
D    178000
E     56000
F     51000
Name: count, dtype: int64

#### we will plot the grades obtained in a order

In [27]:
# STEFANOS: Disable plotting
# sns.countplot(x="Grade", data = df, order=['A','B','C','D','E','F'],  palette="muted")
# plt.show()

In [28]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Grade', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

In [29]:
end_time = time.time()
print(end_time - start_time)

8.505628108978271
